In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
from scipy.interpolate import interp1d

In [ ]:
# -------------------------------------------------
# Fixed-width column definitions
# (0-based indexing: start inclusive, end exclusive)
# -------------------------------------------------
colspecs = [
    (1, 20),     # objID
    (338, 344),  # Number

    (149, 156),  #umag
    (157, 164),  # gmag
    (165, 172),  # rmag
    (173, 180),  # imag
    (181, 188),  # zmag

    (606, 610),  # properH
    (612, 620),  # propera
    (622, 629),  # propere
    (631, 638),  # propersini

    (480, 481),  #bphot_u
    (482, 483),  # bphot_g
    (484, 485),  # bphot_r
    (486, 487),  # bphot_i
    (488, 489),  # bphot_z

    (596, 597),  # grcomplex
]

colnames = [
    "objID","Number",
    "umag", "gmag", "rmag", "imag", "zmag",
    "properH", "propera", "propere", "propersini",
    "bphot_u", "bphot_g", "bphot_r", "bphot_i", "bphot_z", "grcomplex"
]

In [ ]:
# SDSS data file (Sergeyev & Carry (2021) available in http://cdsarc.u-strasbg.fr/viz-bin/cat/J/A+A/652/A59 or cdsarc.u-strasbg.fr (130.79.128.5)

df = pd.read_fwf("sso.dat", colspecs=colspecs, names=colnames)

In [ ]:
#--------------------------------------------------------
# Clean types
#--------------------------------------------------------
df["grcomplex"] = df["grcomplex"].astype(str).str.strip().str.upper()

num_cols = [
    "umag","gmag", "rmag", "imag", "zmag",
    "properH", "propera", "propere", "propersini",
    "bphot_u", "bphot_g", "bphot_r", "bphot_i", "bphot_z"
]

df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

In [ ]:
#---------------------------------------------------
# Filter by complex
#---------------------------------------------------
df = df[df["grcomplex"].isin(["S", "Q", "V"])].copy()
print("After S/Q/V filter:", len(df))

After S/Q/V filter: 780398


In [ ]:
print("bphot_u unique:", df["bphot_u"].value_counts(dropna=False).head(10))
print("bphot_g unique:", df["bphot_g"].value_counts(dropna=False).head(10))
print("bphot_r unique:", df["bphot_r"].value_counts(dropna=False).head(10))
print("bphot_i unique:", df["bphot_i"].value_counts(dropna=False).head(10))


bphot_u unique: bphot_u
1    419440
0    360958
Name: count, dtype: int64
bphot_g unique: bphot_g
1    699982
0     80416
Name: count, dtype: int64
bphot_r unique: bphot_r
1    769696
0     10702
Name: count, dtype: int64
bphot_i unique: bphot_i
1    767876
0     12522
Name: count, dtype: int64


In [ ]:
#---------------------------------------------------
# filterout good g, r, i photometry
#---------------------------------------------------
df = df[
    (df["bphot_u"] == 1) &
    (df["bphot_g"] == 1) &
    (df["bphot_r"] == 1) &
    (df["bphot_i"] == 1) &
    (df["bphot_z"] == 1)
].copy()

print("After photometry + mag filters:", len(df))

After photometry + mag filters: 311112


In [ ]:
#---------------------------------------------------
# Drop missing mandatory magnitudes
#---------------------------------------------------
df = df.dropna(subset=["umag", "gmag", "rmag", "imag"])
print("After dropping NaNs:", len(df))

After dropping NaNs: 311112


In [ ]:
df_sdss = df[
    [
        "objID", "Number", "grcomplex",
        "propera", "propere", "propersini", "properH",
        "umag","gmag", "rmag", "imag", "zmag"
    ]
]

# -------------------------------------------------
# Remove unnumbered asteroids and missing z-band
# -------------------------------------------------
df_sdss["Number"] = (
    df_sdss["Number"]
    .astype(str)
    .str.strip()
)

df_sdss = df_sdss[
    (df_sdss["Number"] != "-") &
    (df_sdss["Number"] != "") &
    (df_sdss["zmag"].notna())
].copy()

# Convert Number to integer (safe now)
df_sdss["Number"] = df_sdss["Number"].astype(int)

print("Remaining asteroids:", len(df_sdss))

#----------------------------------------------
# Export to excel (S / Q / V seperate sheets)
#----------------------------------------------
output_file = "SDSS_SQV_UGRIZ.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for c in ["S", "Q", "V"]:
        df_sdss[df_sdss["grcomplex"] == c].to_excel(
            writer,
            sheet_name=f"{c}_complex",
            index=False
        )

print(f"Excel file written: {output_file}")


/tmp/ipykernel_3041/1871026870.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sdss["Number"] = (


Remaining asteroids: 197987
Excel file written: /content/drive/My Drive/Part3/SDSS/SDSS_SQV_GRIZ.xlsx


In [ ]:
#--------------------------------------------------
# SDSS to reflectance conversion (DeMeo et al. 2009)
#--------------------------------------------------
SOLAR_COLORS = {
    "g": 0.0,
    "r": -0.45,
    "i": -0.55,
    "z": -0.61,
}

def demeo_reflectance(row):
    # g, i, z normalized at r
    Rg = 10 ** (-0.4 * ((row.gmag - row.rmag) -
                        (SOLAR_COLORS["g"] - SOLAR_COLORS["r"])))
    Ri = 10 ** (-0.4 * ((row.imag - row.rmag) -
                        (SOLAR_COLORS["i"] - SOLAR_COLORS["r"])))
    if row.bphot_z == 1 and pd.notna(row.zmag):
        Rz = 10 ** (-0.4 * ((row.zmag - row.rmag) -
                            (SOLAR_COLORS["z"] - SOLAR_COLORS["r"])))
    else:
        Rz = np.nan

    return pd.Series({
        "Rg": Rg,
        "Rr": 1.0,
        "Ri": Ri,
        "Rz": Rz
    })

reflectance = df.apply(demeo_reflectance, axis=1)

In [ ]:
df_final = pd.concat(
    [df.reset_index(drop=True),
     reflectance.reset_index(drop=True)],
    axis=1
)


In [ ]:
df_final = df_final[
    [
        "objID", "Number", "grcomplex",
        "propera", "propere", "propersini", "properH",
        "Rg", "Rr", "Ri", "Rz"
    ]
]

print("Final rows:", len(df_final))
print("Objects with valid z:", df_final["Rz"].notna().sum())

Final rows: 494796
Objects with valid z: 494796


In [ ]:
# -------------------------------------------------
# Remove unnumbered asteroids and missing z-band
# -------------------------------------------------
df_final["Number"] = (
    df_final["Number"]
    .astype(str)
    .str.strip()
)

df_final = df_final[
    (df_final["Number"] != "-") &
    (df_final["Number"] != "") &
    (df_final["Rz"].notna())
].copy()

# Convert Number to integer (safe now)
df_final["Number"] = df_final["Number"].astype(int)

print("Remaining asteroids:", len(df_final))


Remaining asteroids: 266040


In [ ]:
#-------------------------------------------------
#Export to excel (S / Q / V seperate sheets)
#-------------------------------------------------
output_file = "SDSS_SQV_reflectance_GRIZ.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for c in ["S", "Q", "V"]:
        df_final[df_final["grcomplex"] == c].to_excel(
            writer,
            sheet_name=f"{c}_complex",
            index=False
        )

print(f"Excel file written: {output_file}")

Excel file written: /content/drive/My Drive/Part3/SDSS/SDSS_SQV_reflectance_GRIZ_2.xlsx
